# CPS instrument smoke test

Coupling-Phase Spectroscopy — governed Colab runner.

This notebook validates the package, unit tests, and synthetic matrix instrument before any Pythia download.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_perturbations.py", "tests/test_spectra.py", "tests/pythia/test_reduced_operator.py"], check=True)
subprocess.run([sys.executable, "experiments/synthetic_quadratics.py"], check=True)

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)